In [ ]:
from pyspark.sql.functions import col, regexp_replace, when, lit
from pyspark.sql.types import DecimalType

In [ ]:
tabela_origem = "lakehouse_vertice.bronze.financeiro"
tabela_destino = "lakehouse_vertice.silver.financeiro"

df_financeiro = spark.read.table(tabela_origem)

df_financeiro_silver = (
    df_financeiro.withColumn( 'valor',
    regexp_replace(col('valor'), ',', '.').cast(DecimalType(15, 2))
)
    .withColumn( 'categoria_cliente',
    when(col('valor').isNull(), lit('Nao atribuido'))
    .when((col('valor') <= 0), lit('Cliente nao orcado'))
    .when((col('valor') < 10000), lit('Cliente Bronze'))
    .when((col('valor') < 50000), lit('Cliente Prata'))
    .otherwise(lit('Cliente Ouro'))
    )
)

(
df_financeiro_silver.write \
    .format('delta') \
    .mode('overwrite') \
    .saveAsTable(tabela_destino)
)
print(f"Tabela Silver criada/atualizada: {tabela_destino}")